# Session 10 — The Training Loop

A small transformer and a real training loop made observable from first principles. Run this notebook top-to-bottom; every required experiment is implemented **inside this notebook**.

Notation: **B** batch, **T** sequence length, **C** model width, **H** attention heads, **D=C/H** head width, **F** feed-forward width, **V** vocabulary.

## 0. Setup, data, model, and loss

In [1]:
import copy, json, math, os, platform, random, struct, time
from pathlib import Path
import matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as F
SEED=7; random.seed(SEED); torch.manual_seed(SEED); torch.set_num_threads(min(4,os.cpu_count() or 1))
ART=Path('artifacts'); ART.mkdir(exist_ok=True)
PROSE=("Training loops should tell the truth. A model sees tokens, produces logits, compares predictions with targets, and follows gradients. Correct normalization matters. ")*100
DIAG=("ERROR retry=3 code=E17 gradient mismatch inspect shape mask scale 000111. ")*120
chars=sorted(set(PROSE+DIAG)); stoi={c:i for i,c in enumerate(chars)}; V=len(chars)
def enc(s): return torch.tensor([stoi[c] for c in s],dtype=torch.long)
prose,diag,full=enc(PROSE),enc(DIAG),enc(PROSE+DIAG)
def batch(data,B,T):
    starts=torch.randint(0,len(data)-T-1,(B,)); return torch.stack([data[i:i+T] for i in starts]),torch.stack([data[i+1:i+T+1] for i in starts])
class TinyGPT(nn.Module):
    def __init__(self,C=24,H=3,Ff=48,Tmax=64):
        super().__init__(); self.C,self.H,self.D,self.Ff=C,H,C//H,Ff
        self.tok=nn.Embedding(V,C); self.pos=nn.Embedding(Tmax,C); self.ln1=nn.LayerNorm(C); self.qkv=nn.Linear(C,3*C,bias=False); self.proj=nn.Linear(C,C,bias=False); self.ln2=nn.LayerNorm(C); self.fc1=nn.Linear(C,Ff); self.fc2=nn.Linear(Ff,C); self.head=nn.Linear(C,V,bias=False)
    def forward(self,x,trace=False):
        B,T=x.shape; C,H,D=self.C,self.H,self.D; S={}
        def r(n,z): S[n]=tuple(z.shape); return z
        r('tokens',x); z=r('token_embedding',self.tok(x)); p=r('position_ids',torch.arange(T,device=x.device)); z=r('embedded_sum',z+r('position_embedding',self.pos(p)))
        n=r('ln1',self.ln1(z)); packed=r('qkv_packed',self.qkv(n)); q,k,v=packed.chunk(3,-1); r('q',q); r('k',k); r('v',v)
        q=r('q_heads',q.view(B,T,H,D).transpose(1,2)); k=r('k_heads',k.view(B,T,H,D).transpose(1,2)); v=r('v_heads',v.view(B,T,H,D).transpose(1,2)); sc=r('attention_scores',(q@k.transpose(-2,-1))/math.sqrt(D)); m=r('causal_mask',torch.triu(torch.ones(T,T,dtype=torch.bool,device=x.device),1)); a=r('attention_weights',F.softmax(sc.masked_fill(m,float('-inf')),-1)); h=r('context_heads',a@v); h=r('context',h.transpose(1,2).contiguous().view(B,T,C)); z=r('residual_after_attention',z+r('attention_output',self.proj(h))); n=r('ln2',self.ln2(z)); h=r('mlp_hidden',self.fc1(n)); h=r('mlp_activation',F.gelu(h)); z=r('residual_after_mlp',z+r('mlp_output',self.fc2(h))); out=r('logits',self.head(z)); return (out,S) if trace else out
def lm_loss(m,x,y,reduction='mean'):
    z=m(x); return F.cross_entropy(z.reshape(-1,z.shape[-1]),y.reshape(-1),reduction=reduction)
model=TinyGPT(); print('vocab',V,'parameters',sum(p.numel() for p in model.parameters()),'torch',torch.__version__)

## 1. Print every tensor shape

Each printed line includes both the concrete shape and what its dimensions mean. The same step also prints the target, flattened loss tensors, scalar loss, and every parameter/gradient shape.

In [2]:
x,y=batch(prose,4,24); m=copy.deepcopy(model); opt=torch.optim.SGD(m.parameters(),lr=.01); opt.zero_grad(); logits,S=m(x,True)
meaning={'tokens':'[B,T] token ids','token_embedding':'[B,T,C] token vectors','position_ids':'[T] positions','position_embedding':'[T,C] position vectors','embedded_sum':'[B,T,C] embedding sum','ln1':'[B,T,C] normalized stream','qkv_packed':'[B,T,3C] packed QKV','q':'[B,T,C] queries','k':'[B,T,C] keys','v':'[B,T,C] values','q_heads':'[B,H,T,D] query heads','k_heads':'[B,H,T,D] key heads','v_heads':'[B,H,T,D] value heads','attention_scores':'[B,H,T,T] query-key scores','causal_mask':'[T,T] future mask','attention_weights':'[B,H,T,T] probabilities','context_heads':'[B,H,T,D] attended values','context':'[B,T,C] concatenated heads','attention_output':'[B,T,C] attention projection','residual_after_attention':'[B,T,C] residual stream','ln2':'[B,T,C] normalized stream','mlp_hidden':'[B,T,F] MLP expansion','mlp_activation':'[B,T,F] GELU','mlp_output':'[B,T,C] MLP projection','residual_after_mlp':'[B,T,C] final hidden','logits':'[B,T,V] vocabulary scores'}
print('B=batch T=sequence C=model width H=heads D=C/H F=feed-forward width V=vocab')
for n,s in S.items(): print(f'{n:26s} {str(s):16s} {meaning[n]}')
flat=logits.reshape(-1,V); targ=y.reshape(-1); loss=F.cross_entropy(flat,targ); print('targets',tuple(y.shape),'[B,T]'); print('flat_logits',tuple(flat.shape),'[B*T,V]'); print('flat_targets',tuple(targ.shape),'[B*T]'); print('loss',tuple(loss.shape),'scalar'); loss.backward()
for n,p in m.named_parameters(): print(f'{n:28s} param={tuple(p.shape)} grad={tuple(p.grad.shape)}')
opt.step(); opt.zero_grad(set_to_none=True)

## 2. Verify one gradient by hand

For one scalar weight,
\[
\frac{dL}{dw}\approx\frac{L(w+\epsilon)-L(w-\epsilon)}{2\epsilon}.
\]
The finite-difference calculation is independent of `backward()` and uses float64 to make the diagnostic precise.

In [3]:
torch.manual_seed(SEED+20); g=copy.deepcopy(model).double(); gx,gy=batch(prose,2,10); g.zero_grad(); L=lm_loss(g,gx,gy); L.backward(); G=g.head.weight.grad; idx=G.abs().argmax().item(); row,col=idx//G.shape[1],idx%G.shape[1]; auto=G[row,col].item(); eps=1e-5
with torch.no_grad():
    w0=g.head.weight[row,col].item(); g.head.weight[row,col]=w0+eps; lp=lm_loss(g,gx,gy).item(); g.head.weight[row,col]=w0-eps; lm=lm_loss(g,gx,gy).item(); g.head.weight[row,col]=w0
finite=(lp-lm)/(2*eps); rel=abs(auto-finite)/(max(abs(auto),abs(finite),1e-12)); print('weight',row,col,'backward',auto,'finite_difference',finite,'relative_error',rel); assert rel<1e-6

## 3. Break gradient accumulation on purpose

The short micro-batch has **8×12 = 96 tokens** and the long micro-batch has **8×48 = 384 tokens**. The wrong loop averages two already-averaged losses, giving each micro-batch 50% weight. The correct loop sums token losses and divides once by the total token count, giving 20%/80% weight. Both curves are evaluated with the correct token-weighted objective and plotted together.

In [4]:
torch.manual_seed(SEED+1); base=TinyGPT(); wrong,correct=copy.deepcopy(base),copy.deepcopy(base); ow=torch.optim.SGD(wrong.parameters(),lr=.15); oc=torch.optim.SGD(correct.parameters(),lr=.15); exs,eys=batch(diag,8,12); exl,eyl=batch(prose,8,48)
def ev(m):
    with torch.no_grad(): return ((lm_loss(m,exs,eys,'sum')+lm_loss(m,exl,eyl,'sum'))/(eys.numel()+eyl.numel())).item()
wc,cc=[],[]
for step in range(60):
    sx,sy=batch(diag,8,12); lx,ly=batch(prose,8,48)
    ow.zero_grad(); ((lm_loss(wrong,sx,sy)+lm_loss(wrong,lx,ly))/2).backward(); ow.step()
    oc.zero_grad(); ((lm_loss(correct,sx,sy,'sum')+lm_loss(correct,lx,ly,'sum'))/(sy.numel()+ly.numel())).backward(); oc.step(); wc.append(ev(wrong)); cc.append(ev(correct))
gap=max(abs(a-b) for a,b in zip(wc,cc)); print('tokens',eys.numel(),eyl.numel(),'max curve gap',gap,'final wrong/correct',wc[-1],cc[-1]); assert gap>1e-3
plt.figure(figsize=(8,4)); plt.plot(wc,label='WRONG average-of-averages'); plt.plot(cc,label='CORRECT token-weighted'); plt.xlabel('optimizer update'); plt.ylabel('fixed token-weighted loss'); plt.legend(); plt.tight_layout(); plt.savefig(ART/'gradient_accumulation_wrong_vs_correct.png',dpi=150); plt.show()
# Stronger identity proof on same-distribution unequal-length batches.
torch.manual_seed(SEED+11); b=TinyGPT(); sx,sy=batch(prose,8,12); lx,ly=batch(prose,8,48); N=sy.numel()+ly.numel()
def fg(m): return torch.cat([p.grad.reshape(-1).float() for p in m.parameters() if p.grad is not None])
r=copy.deepcopy(b); r.zero_grad(); ((lm_loss(r,sx,sy,'sum')+lm_loss(r,lx,ly,'sum'))/N).backward(); gr=fg(r)
c=copy.deepcopy(b); c.zero_grad(); (lm_loss(c,sx,sy,'sum')/N).backward(); (lm_loss(c,lx,ly,'sum')/N).backward(); gc=fg(c)
w=copy.deepcopy(b); w.zero_grad(); (.5*lm_loss(w,sx,sy)).backward(); (.5*lm_loss(w,lx,ly)).backward(); gw=fg(w)
crel=(gc-gr).norm().item()/(gr.norm().item()+1e-12); wrel=(gw-gr).norm().item()/(gr.norm().item()+1e-12); print('reference proof correct rel-L2',crel,'broken rel-L2',wrel); assert crel<1e-6 and wrel>1e-3

## 4. Log grad norm at every step

The loop logs **every** global L2 gradient norm together with a fixed probe loss. Then it finds a step where the gradient norm changes much more than the same-step loss; the following few updates show the later loss movement.

In [5]:
def gn(m): return math.sqrt(sum(p.grad.detach().float().norm().item()**2 for p in m.parameters() if p.grad is not None))
torch.manual_seed(SEED+2); t=TinyGPT(); o=torch.optim.AdamW(t.parameters(),lr=3e-3); px,py=batch(prose,8,32); losses=[]; norms=[]
for step in range(60):
    bx,by=batch(full,8,32); o.zero_grad(); l=lm_loss(t,bx,by); l.backward(); n=gn(t); o.step();
    with torch.no_grad(): pl=lm_loss(t,px,py).item()
    losses.append(pl); norms.append(n); print(f'step={step:02d} probe_loss={pl:.6f} grad_norm={n:.6f}')
cand=[]
for i in range(1,56):
    gm=abs(norms[i]-norms[i-1])/(abs(norms[i-1])+1e-12); lm0=abs(losses[i]-losses[i-1])/(abs(losses[i-1])+1e-12); fut=max(abs(losses[j]-losses[i])/(abs(losses[i])+1e-12) for j in range(i+1,i+4)); cand.append((gm/(lm0+1e-5)*fut,i,gm,lm0,fut))
_,lead,gm,lm0,fut=max(cand); print('leading step',lead,'grad move',gm,'same-step loss move',lm0,'next-3 loss move',fut); assert gm>lm0
fig,ax=plt.subplots(figsize=(8,4)); ax.plot(losses,label='probe loss'); ax.axvline(lead,ls='--'); ax2=ax.twinx(); ax2.plot(norms,alpha=.65,label='grad norm'); fig.tight_layout(); fig.savefig(ART/'grad_norm_leading_signal.png',dpi=150); plt.show()

## 5. Compute my own MFU

For this one-block transformer, dominant forward matmul FLOPs/token are approximated as
\[
8C^2+4CF+2CV+4TC,
\]
and training is approximated as 3× forward for forward+backward.

\[
MFU=\frac{\text{estimated model FLOPs/s}}{\text{peak FLOPs/s}}.
\]

On this CPU run the denominator is an **explicit analytical allocated-core FP32 roofline**, not the measured GEMM rate. The estimate uses PyTorch thread count × detected GHz × SIMD fp32 lanes × 2 FLOPs/FMA × assumed vector issue units. On a GPU, set `THEORETICAL_PEAK_TFLOPS` to the vendor peak for the exact precision.

In [6]:
def sync():
    if torch.cuda.is_available(): torch.cuda.synchronize()
def bench(m,B=16,T=32,iters=15):
    opt=torch.optim.SGD(m.parameters(),lr=1e-3); x,y=batch(prose,B,T); dev=next(m.parameters()).device; x,y=x.to(dev),y.to(dev)
    for _ in range(4): opt.zero_grad(); z=lm_loss(m,x,y); z.backward(); opt.step()
    sync(); start=time.perf_counter()
    for _ in range(iters): opt.zero_grad(); z=lm_loss(m,x,y); z.backward(); opt.step()
    sync(); return (time.perf_counter()-start)/iters,x.numel(),T
def cpu_peak():
    text=Path('/proc/cpuinfo').read_text(errors='ignore') if Path('/proc/cpuinfo').exists() else ''; flags=set(); mhz=2300.
    for line in text.splitlines():
        if line.startswith('flags') and not flags: flags=set(line.split(':',1)[1].split())
        if line.startswith('cpu MHz'): mhz=float(line.split(':',1)[1]); break
    lanes=16 if 'avx512f' in flags else 8 if ('avx2' in flags or 'avx' in flags) else 4; units=2 if 'fma' in flags else 1; peak=torch.get_num_threads()*mhz*1e6*lanes*2*units/1e12; return peak,f'{torch.get_num_threads()} threads × {mhz/1000:.3f} GHz × {lanes} fp32 lanes × 2 FLOP/FMA × {units} vector units'
dev=torch.device('cuda' if torch.cuda.is_available() else 'cpu'); pm=TinyGPT().to(dev); C,Ff=pm.C,pm.Ff
def ftok(T): return 3*(8*C*C+4*C*Ff+2*C*V+4*T*C)
sec,tok,T=bench(pm); tflops=ftok(T)*tok/sec/1e12; THEORETICAL_PEAK_TFLOPS=None
if dev.type=='cpu': peak,source=cpu_peak()
elif THEORETICAL_PEAK_TFLOPS: peak,source=float(THEORETICAL_PEAK_TFLOPS),'configured vendor peak'
else: peak,source=None,'vendor peak required'
mfu=tflops/peak if peak else None; gap40=(.40-mfu)*100 if mfu is not None else None; print('step seconds',sec,'tokens/s',tok/sec,'model TFLOP/s',tflops,'peak',peak,'MFU',mfu,'gap to 40pp',gap40,'denominator',source)
print('Why below 40%: tiny GEMMs, eager/Python dispatch, softmax/layernorm/optimizer and memory costs not fully credited in the simplified FLOP numerator, no fusion/compile, and small batch/sequence shapes.')

## 6. Write 0.1 by hand: fp32, bf16, fp8 E4M3

\[
0.1_{10}=0.00011001100110011\ldots_2=1.1001100110011\ldots_2\times2^{-4}.
\]
The `0011` tail repeats forever.

**fp32:** sign `0`; exponent `-4+127=123=01111011`; fraction starts `100110011...`; keep 23 bits `10011001100110011001100`; discarded tail starts with `1` and is non-zero, so round-to-nearest-even to `10011001100110011001101`. Final bits: **`0 | 01111011 | 10011001100110011001101` = `0x3DCCCCCD`**.

**bf16:** sign `0`; exponent `01111011`; keep 7 fraction bits `1001100`; next bit is `1` with non-zero tail, so round to `1001101`. Final bits: **`0 | 01111011 | 1001101` = `0x3DCD`**.

**fp8 E4M3FN (bias 7):** sign `0`; exponent `-4+7=3=0011`; keep 3 fraction bits `100`; discarded `110...` is greater than half an ULP, so round to `101`. Final bits: **`0 | 0011 | 101` = `0x1D`**.

I would train in **BF16** on native BF16 hardware: it keeps the 8-bit exponent range of fp32 while reducing memory/bandwidth. Naive E4M3 is too coarse for all training state without scaling and higher-precision accumulation.

In [7]:
fp32=struct.unpack('>I',struct.pack('>f',.1))[0]; bf=torch.tensor(.1,dtype=torch.bfloat16); bf16=int(bf.view(torch.uint16)); fp8=torch.tensor(.1,dtype=torch.float8_e4m3fn) if hasattr(torch,'float8_e4m3fn') else None; e4=int(fp8.view(torch.uint8)) if fp8 is not None else 0x1D
print('fp32',f'{fp32:032b}',hex(fp32)); print('bf16',f'{bf16:016b}',hex(bf16)); print('E4M3',f'{e4:08b}',hex(e4)); assert fp32==0x3DCCCCCD and bf16==0x3DCD and e4==0x1D
summary={'schema_version':3,'gradient_check':{'relative_error':rel,'autograd':auto,'finite_difference':finite},'gradient_accumulation':{'short_tokens':96,'long_tokens':384,'max_curve_gap':gap,'correct_reference_rel_l2':crel,'broken_reference_rel_l2':wrel},'grad_norm':{'steps':len(norms),'leading_step':lead,'grad_move':gm,'same_step_loss_move':lm0,'next_3_loss_move':fut},'mfu':{'device':str(dev),'seconds_per_step':sec,'tokens_per_second':tok/sec,'estimated_model_tflops':tflops,'peak_tflops':peak,'peak_source':source,'reported_mfu':mfu,'gap_to_40_percentage_points':gap40},'float_0_1':{'fp32_bits':f'{fp32:032b}','bf16_bits':f'{bf16:016b}','fp8_e4m3_bits':f'{e4:08b}','training_choice':'bf16'}}
(ART/'metrics.json').write_text(json.dumps(summary,indent=2)); print(json.dumps(summary,indent=2)); print('All assignment checks passed.')

## Final finding

The training loop is now falsifiable: tensor axes are explicit, one derivative is independently checked, incorrect accumulation visibly diverges from correct token weighting, grad norm is logged every step, MFU exposes its denominator and gap to 40%, and the 0.1 bit patterns are derived manually and verified programmatically.